# OD 집계와 PageRank 순위 비교
소형 테스트 fixture를 사용하는 교육용 재현 예제입니다. 실제 서울·대전·세종 분석 수치로 인용하지 않습니다.
입력: backend/tests/fixtures/od_daily.csv. 2026-10의 A/B/C 등 테스트 ID이며 기존 수업 보고서의 실측 자료가 아닙니다.


In [ ]:
from pathlib import Path
import pandas as pd
import networkx as nx
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'backend/tests/fixtures/od_daily.csv').exists())
od = pd.read_csv(root / 'backend/tests/fixtures/od_daily.csv')
od.head()


## 도시 ID를 분리하고 이동량을 가중치로 집계
도시가 다른 같은 ID를 하나의 노드로 합치지 않습니다. 가중치는 이동 건수입니다.


In [ ]:
city = 'seoul'
edges = od[od.region_id.eq(city)].groupby(['origin_station_id', 'destination_station_id'], as_index=False).trip_count.sum()
graph = nx.from_pandas_edgelist(edges, 'origin_station_id', 'destination_station_id', edge_attr='trip_count', create_using=nx.DiGraph())
scores = nx.pagerank(graph, weight='trip_count')
rentals = edges.groupby('origin_station_id').trip_count.sum()
comparison = pd.DataFrame({'rentals': rentals, 'pagerank': pd.Series(scores)}).fillna({'rentals': 0})
comparison['rental_rank'] = comparison.rentals.rank(method='min', ascending=False)
comparison['pagerank_rank'] = comparison.pagerank.rank(method='min', ascending=False)
comparison['rank_gap'] = comparison.rental_rank - comparison.pagerank_rank
assert int(comparison.rentals.sum()) == int(edges.trip_count.sum())
assert abs(comparison.pagerank.sum() - 1) < 1e-6
comparison.sort_values('pagerank', ascending=False)


## 해석과 한계
대여량은 출발 이동의 총량이고 PageRank는 방향·가중치를 반영한 연결 중요도입니다. 순위 차이가 설치 효과나 환승 수요를 증명하지 않습니다.
실제 분석으로 확장할 때는 같은 도시·기간의 검증된 OD, 대여소 마스터, 제외 규칙을 사용하고 날짜 범위와 출처를 함께 기록하십시오.
